# 04L. 로컬용: 윈도우 → 문장 → train_pool / val / test + k-shot 파일 (A, B)

`03_window_to_text.ipynb` + `04_split.ipynb`를 **로컬 데이터 그대로** 한 번에 돌리도록 합친 노트북. 규칙과 결과 파일 형식은 04와 같다. 05(`finetuning.ipynb`)가 이 결과를 그대로 읽는다.

**03·04와 다른 점**

| | 03·04 | 이 노트북 |
|---|---|---|
| 입력 | 드라이브의 zip, `processed/*.jsonl` | 로컬 `data/interim/windows_{A,B}_size5_stride1.csv` (`src/windows.py` 결과) |
| 출력 | 드라이브 `splits/` | 로컬 `data/splits/{A,B}/` (git에서 무시됨) |
| 읽기/쓰기 표시 | `readOnly`가 비면 무조건 `write` | 비면 API 이름으로 추정 (아래 2번). BOTS v3는 옛 CloudTrail 형식이라 95%가 비어 있어서, 그대로 두면 정상 조회가 전부 `write`로 적힘 |
| 데이터 | 정상 세션 2개 (Splunk) | 정상 세션 3개 (Splunk 2 + BOTS v3 1) |

**출력** (`data/splits/{A,B}/`, 04와 같은 이름)
- `pool.jsonl`, `val.jsonl`, `test.jsonl`
- `kshot/train_k{k}_s{seed}.json` (k = 1, 3, 5, 8, 16 × seed 0, 1, 2)
- `val_gliclass.json` (후보 라벨 = 학습 5개), `test_gliclass.json` (학습 5 + 숨긴 3)
- `data/processed/windows_{A,B}.jsonl` (03의 출력과 같은 형식), `data/labels.csv` (05가 찾는 라벨 표 복사본)

**나누는 규칙 (04와 같음)**
1. 숨긴 전술(Defense Impairment, Impact, Collection)이 붙은 윈도우는 전부 test
2. 윈도우 20개 이상인 큰 세션은 세션 안에서 시간 순서대로 앞 50% pool, 다음 25% val, 마지막 25% test
3. 20개 미만인 작은 세션은 세션 통째로 한 곳에 (라벨별 50/25/25에 가깝게)
4. 같은 세션에서 다른 칸으로 간 윈도우끼리 이벤트가 겹치면(윈도우 번호 차이 5 미만) 아래 칸 쪽을 버림 (test > val > pool)

**준비**: `labels/build_labels.py` → `notebooks/preprocess.ipynb` → `src/windows.py`를 먼저 돌려 `data/interim/`에 윈도우 파일이 있어야 함. CPU로 1~2분.

## 0. 경로

In [1]:
import os, json, shutil
from pathlib import Path
import pandas as pd, numpy as np
from collections import Counter
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 200)

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = REPO / 'data'
INTERIM, PROC, SPLITS = DATA / 'interim', DATA / 'processed', DATA / 'splits'
LABELS = REPO / 'labels' / 'labels.csv'
for k in 'AB':
    assert (INTERIM / f'windows_{k}_size5_stride1.csv').exists(), f'windows_{k} 없음 (src/windows.py 먼저)'
assert LABELS.exists(), 'labels/labels.csv 없음'
PROC.mkdir(parents=True, exist_ok=True); SPLITS.mkdir(parents=True, exist_ok=True)
shutil.copy(LABELS, DATA / 'labels.csv')          # 05가 data/ 아래에서 labels.csv를 찾음
print('REPO =', REPO)

REPO = /home/openclaw/work2/ctgli


## 1. 불러오기, 확정 라벨 적용 (03의 1번과 같은 규칙)

[하는 일] 원래 전술 → 우리 라벨(`labels.csv`의 `source_tactics`), 제외 라벨 윈도우 삭제, 같은 signature는 하나만, 우리 라벨 기준 충돌 signature는 삭제.

[볼 것] A, B 남은 윈도우 수.

In [2]:
wins = {k: pd.read_csv(INTERIM / f'windows_{k}_size5_stride1.csv', low_memory=False, keep_default_na=False) for k in 'AB'}
labels = pd.read_csv(LABELS)
to_label = {t: r.label_id for r in labels.itertuples() for t in str(r.source_tactics).split('|') if t and t != 'nan'}
role = dict(zip(labels.label_id, labels.role))
TRAIN = labels[labels.role == 'train'].label_id.tolist()
HELD = labels[labels.role == 'held_out'].label_id.tolist()
name = dict(zip(labels.label_id, labels.name))
text_of = dict(zip(labels.label_id, labels.label_text))

def apply_labels(k):
    u = wins[k].copy()
    is_att = u.m1_is_attack.astype(str) == 'True'
    tac = u.m1_tactics.where(is_att, '').fillna('')
    u['labels'] = tac.apply(lambda s: sorted({to_label[t] for t in s.split('|') if t}) or ['BENIGN'])
    u = u[~u['labels'].apply(lambda ls: any(role[l] == 'excluded' for l in ls))]
    u['key'] = u['labels'].apply('|'.join)
    n_lab = u.groupby('signature').key.nunique()
    u = u[~u.signature.isin(n_lab[n_lab > 1].index)].drop_duplicates('signature')
    return u.reset_index(drop=True)

final = {k: apply_labels(k) for k in 'AB'}
print({k: len(v) for k, v in final.items()})
print('학습 라벨:', TRAIN); print('숨긴 라벨:', HELD)

{'A': 9269, 'B': 7721}
학습 라벨: ['TA0007', 'TA0003+TA0004', 'TA0001+TA0005', 'TA0006', 'BENIGN']
숨긴 라벨: ['TA0112', 'TA0040', 'TA0009']


## 2. 문장 변환 (03의 2번 + 읽기/쓰기 추정)

[하는 일] signature를 영어 문장으로 바꾼다. 문장 규칙은 03과 같다.

**바뀐 점**: `readOnly`가 비어 있으면 API 이름 앞부분으로 읽기/쓰기를 정한다 (`Describe`, `List`, `Get` 등 → read, 나머지 → write). 03은 비어 있으면 무조건 write였다.

[볼 것] BOTS v3 정상 윈도우가 `read`로 바뀌었는지, 소스별 read 비율이 비슷해졌는지.

[확인 질문] 왜 이게 정답 누수와 관련이 있을까? (힌트: 정상 윈도우의 3분의 2가 BOTS v3)

In [3]:
READ_PREFIX = ('Describe', 'List', 'Get', 'Lookup', 'Head', 'Search', 'Check', 'BatchGet', 'Scan', 'Query', 'Select')

def is_read(ro, api):
    if ro in ('True', 'False'):
        return ro == 'True'
    return api.startswith(READ_PREFIX)             # readOnly가 없는 옛 CloudTrail 형식

def parse(sig):
    out = []
    for e in sig.split(' > '):
        src, api, err, ro, utype, reg, rep, actor = e.split('|')
        out.append(dict(svc=src.split('.')[0], name=api, err=err, read=is_read(ro, api),
                        utype=utype or 'unknown', region=(reg == 'True'), rep=int(float(rep or 1)), actor=actor))
    return out

def to_text(sig):
    ev = parse(sig)
    actors = list(dict.fromkeys(e['actor'] for e in ev))
    utypes = list(dict.fromkeys(e['utype'] for e in ev))
    fails = [e for e in ev if e['err']]
    head = (f"CloudTrail window: {len(ev)} call group{'s' if len(ev) > 1 else ''} by {len(actors)} actor"
            f"{'s' if len(actors) > 1 else ''} ({', '.join(utypes)}), "
            f"{'%d failed' % len(fails) if fails else 'no failures'}, "
            f"region {'changed' if any(e['region'] for e in ev) else 'unchanged'}.")
    steps, prev = [], None
    for e in ev:
        tags = ['read' if e['read'] else 'write']
        if e['rep'] > 1: tags.append(f"x{e['rep']}")
        if e['err']: tags.append(f"failed: {e['err']}")
        who = f"[{e['actor']}] " if prev is not None and e['actor'] != prev else ''
        steps.append(f"{who}{e['svc']} {e['name']} ({', '.join(tags)})")
        prev = e['actor']
    return head + ' Sequence: ' + ' -> '.join(steps)

def read_share(sig, fixed):
    ev = [e.split('|') for e in sig.split(' > ')]
    return np.mean([is_read(f[3], f[1]) if fixed else f[3] == 'True' for f in ev])

d = final['B']
print('소스별 read 비율 (B, 정상 윈도우):')
print(pd.DataFrame({'03 방식': d[d.key == 'BENIGN'].groupby('source').signature.apply(lambda s: s.map(lambda x: read_share(x, False)).mean()),
                    '이 노트북': d[d.key == 'BENIGN'].groupby('source').signature.apply(lambda s: s.map(lambda x: read_share(x, True)).mean())}).round(2))
ex = d[(d.source == 'botsv3') & (d.key == 'BENIGN')].iloc[0]
print('\n예시 (BOTS v3 정상):', to_text(ex.signature)[:300])

소스별 read 비율 (B, 정상 윈도우):


        03 방식  이 노트북
source              
botsv3   0.05   0.95
splunk   0.95   0.95

예시 (BOTS v3 정상): CloudTrail window: 5 call groups by 4 actors (AssumedRole, AWSService, IAMUser), no failures, region changed. Sequence: kms Decrypt (read, x2) -> [actor1] ec2 DescribeInstanceStatus (read, x10) -> [actor2] sts AssumeRole (write) -> [actor3] sts GetCallerIdentity (read, x4) -> config DescribeConfigRu


## 3. 전체 변환 + 같은 문장 합치기 (03의 3번과 같음)

[하는 일] 문장이 같고 라벨도 같으면 하나로 합치고, 문장이 같은데 라벨이 다르면 뺀다.

[볼 것] 라벨별 예시 문장.

In [4]:
for k in 'AB':
    f = final[k]
    f['text'] = f.signature.apply(to_text)
    n_lab = f.groupby('text').key.nunique()
    bad = n_lab[n_lab > 1].index
    before = len(f)
    final[k] = f[~f.text.isin(bad)].drop_duplicates('text').reset_index(drop=True)
    print(f'[{k}] {before} -> {len(final[k])} (같은 문장 합침, 라벨 충돌 문장 {len(bad)}개 삭제)')
print()
for lid in TRAIN + HELD:
    hit = final['B'][final['B']['labels'].apply(lambda ls: lid in ls)]
    if len(hit):
        print(f'■ {name[lid]} ({lid}) - B {len(hit)}개 중 예시')
        print('  ', hit.text.iloc[len(hit) // 2][:300], '\n')

cols = ['text', 'labels', 'session', 'window', 'source', 'n_attack_events']
for k in 'AB':
    final[k][cols].to_json(PROC / f'windows_{k}.jsonl', orient='records', lines=True, force_ascii=False)
print('저장:', sorted(os.listdir(PROC)))

[A] 9269 -> 8948 (같은 문장 합침, 라벨 충돌 문장 1개 삭제)
[B] 7721 -> 7665 (같은 문장 합침, 라벨 충돌 문장 0개 삭제)

■ Discovery (TA0007) - B 2482개 중 예시
   CloudTrail window: 5 call groups by 2 actors (IAMUser), no failures, region changed. Sequence: iam ListRoles (read, x2) -> iam ListRoles (read, x2) -> iam ListRoles (read, x2) -> [actor1] s3 ListObjects (read, x13) -> [actor0] iam ListRoles (read, x4) 

■ Persistence and Privilege Escalation (TA0003+TA0004) - B 287개 중 예시
   CloudTrail window: 5 call groups by 1 actor (IAMUser), 2 failed, region unchanged. Sequence: iam DeletePolicy (write) -> iam DeletePolicy (write) -> iam DeletePolicy (write, failed: DeleteConflictException) -> iam DeletePolicy (write, x2) -> iam DeletePolicy (write, x2, failed: NoSuchEntityException 

■ Initial Access and Stealth (TA0001+TA0005) - B 135개 중 예시
   CloudTrail window: 5 call groups by 1 actor (SAMLUser), no failures, region unchanged. Sequence: sts AssumeRoleWithSAML (read) -> sts AssumeRoleWithSAML (read, x2) -> sts AssumeRole

저장: ['windows_A.jsonl', 'windows_B.jsonl']


## 4. 세션 구조 보기 (04의 2번과 같음)

[볼 것] 라벨마다 세션 수와 가장 큰 세션 크기. 큰 세션은 세션 안에서 시간으로 나눈다.

In [5]:
data = {k: final[k].copy() for k in 'AB'}
for k in 'AB':
    d = data[k]
    rows = []
    for lid in TRAIN + HELD:
        h = d[d['labels'].apply(lambda ls: lid in ls)]
        top = h.session.value_counts()
        rows.append(dict(label=lid, name=name[lid], role='train' if lid in TRAIN else 'held_out',
                         windows=len(h), sessions=len(top), biggest=top.iloc[0] if len(top) else 0))
    print(f'===== {k}'); print(pd.DataFrame(rows).to_string(index=False)); print()
print('라벨 조합별 개수 (B):'); print(data['B'].key.value_counts().to_string())

===== A
        label                                 name     role  windows  sessions  biggest
       TA0007                            Discovery    train     2722         7     1082
TA0003+TA0004 Persistence and Privilege Escalation    train      218        19       73
TA0001+TA0005           Initial Access and Stealth    train      115         7       58
       TA0006                    Credential Access    train       36        11        9
       BENIGN                               Benign    train     5588         3     3521
       TA0112                   Defense Impairment held_out       72        17       44
       TA0040                               Impact held_out      455         6      425
       TA0009                           Collection held_out       26         3       22

===== B
        label                                 name     role  windows  sessions  biggest
       TA0007                            Discovery    train     2482         7      905
TA0003+TA0004 P

## 5. 나누기 (규칙 1~3, 04의 3번과 같음)

In [6]:
RATIO = {'pool': .50, 'val': .25, 'test': .25}
BIG = 20

def split(d):
    d = d.copy(); d['split'] = None
    held = d['labels'].apply(lambda ls: any(l in HELD for l in ls))
    d.loc[held, 'split'] = 'test'                                   # 규칙 1

    rest = d[~held]
    size = rest.groupby('session').size()
    for s in size[size >= BIG].index:                                # 규칙 2
        idx = rest[rest.session == s].sort_values('window').index
        c1, c2 = int(len(idx) * .50), int(len(idx) * .75)
        d.loc[idx[:c1], 'split'] = 'pool'; d.loc[idx[c1:c2], 'split'] = 'val'; d.loc[idx[c2:], 'split'] = 'test'

    small = size[size < BIG].sort_values(ascending=False).index      # 규칙 3
    main_key = rest.groupby('session').key.agg(lambda s: s.value_counts().index[0])
    total = rest.key.value_counts()
    for s in small:
        key = main_key[s]
        now = d[(d.key == key) & d.split.notna()].split.value_counts()
        need = {sp: RATIO[sp] * total[key] - now.get(sp, 0) for sp in RATIO}
        d.loc[(d.session == s) & d.split.isna(), 'split'] = max(need, key=need.get)
    assert d.split.notna().all()
    return d

parts = {k: split(data[k]) for k in 'AB'}
for k in 'AB':
    print(k, parts[k].split.value_counts().to_dict())

A {'pool': 4201, 'test': 2650, 'val': 2097}
B {'pool': 3550, 'test': 2338, 'val': 1777}


## 6. 겹치는 윈도우 버리기 (규칙 4, 04의 4번과 같음)

In [7]:
RANK = {'pool': 0, 'val': 1, 'test': 2}

def purge(d):
    drop = []
    for s, g in d.groupby('session'):
        w = g.window.values; r = g.split.map(RANK).values
        for i, ix in enumerate(g.index):
            if ((np.abs(w - w[i]) < 5) & (r > r[i])).any():
                drop.append(ix)
    print(f'  겹쳐서 버림: {len(drop)}개', dict(d.loc[drop, 'split'].value_counts()))
    return d.drop(drop).reset_index(drop=True)

for k in 'AB':
    print(k); parts[k] = purge(parts[k])

A
  겹쳐서 버림: 88개 {'pool': np.int64(45), 'val': np.int64(43)}
B


  겹쳐서 버림: 139개 {'pool': np.int64(72), 'val': np.int64(67)}


## 7. 결과 표 + 검사 (04의 5번과 같음)

[볼 것] 학습 라벨은 pool에 16개 이상이어야 k=16 가능 (`max_k`). 숨긴 라벨은 pool, val에 0개.

In [8]:
for k in 'AB':
    d = parts[k]
    t = d.explode('labels').groupby(['labels', 'split']).size().unstack(fill_value=0).reindex(columns=['pool', 'val', 'test'], fill_value=0)
    t = t.reindex(TRAIN + HELD).fillna(0).astype(int)
    t.insert(0, 'name', [name[l] for l in t.index])
    t['max_k'] = [max([kk for kk in [1, 3, 5, 8, 16] if kk <= p], default=0) if l in TRAIN else '-' for l, p in zip(t.index, t.pool)]
    print(f'===== {k}  (총 {len(d)})'); print(t.to_string()); print()

    for sp in ['pool', 'val']:
        assert not d[d.split == sp]['labels'].apply(lambda ls: any(l in HELD for l in ls)).any(), f'{sp}에 숨긴 라벨!'
    assert d.text.is_unique
    for s, g in d.groupby('session'):
        w = g.sort_values('window'); sp = w.split.values; ww = w.window.values
        for i in range(len(w)):
            assert not ((np.abs(ww - ww[i]) < 5) & (sp != sp[i])).any(), f'겹침 남음: {s}'
print('검사 통과: 숨긴 라벨 누수 없음, 문장 중복 없음, 칸 사이 이벤트 겹침 없음')

print('\n칸별 소스 구성 (정상 윈도우):')
for k in 'AB':
    b = parts[k][parts[k].key == 'BENIGN']
    print(k, b.groupby(['split', 'source']).size().unstack(fill_value=0).to_dict('index'))

===== A  (총 8860)
split                                          name  pool   val  test max_k
labels                                                                     
TA0007                                    Discovery  1292   639   766    16
TA0003+TA0004  Persistence and Privilege Escalation    78    34    89    16
TA0001+TA0005            Initial Access and Stealth    31     8    63    16
TA0006                            Credential Access    18     9     9    16
BENIGN                                       Benign  2768  1372  1399    16
TA0112                           Defense Impairment     0     0    72     -
TA0040                                       Impact     0     0   455     -
TA0009                                   Collection     0     0    26     -



===== B  (총 7526)
split                                          name  pool   val  test max_k
labels                                                                     
TA0007                                    Discovery  1173   580   703    16
TA0003+TA0004  Persistence and Privilege Escalation   100    47   107    16
TA0001+TA0005            Initial Access and Stealth    35    18    69    16
TA0006                            Credential Access     6     3     3     5
BENIGN                                       Benign  2199  1080  1124    16
TA0112                           Defense Impairment     0     0    70     -
TA0040                                       Impact     0     0   447     -
TA0009                                   Collection     0     0    49     -



검사 통과: 숨긴 라벨 누수 없음, 문장 중복 없음, 칸 사이 이벤트 겹침 없음

칸별 소스 구성 (정상 윈도우):
A {'pool': {'botsv3': 1745, 'splunk': 1023}, 'test': {'botsv3': 882, 'splunk': 517}, 'val': {'botsv3': 880, 'splunk': 492}}
B {'pool': {'botsv3': 1475, 'splunk': 724}, 'test': {'botsv3': 753, 'splunk': 371}, 'val': {'botsv3': 748, 'splunk': 332}}


## 8. k-shot 학습 파일 뽑기 (04의 6번과 같음)

pool에서 라벨마다 k개씩, seed(0, 1, 2)마다 다르게. 드문 라벨부터 담는다. pool에 k개가 안 되는 라벨이 있으면 그 k는 만들지 않고 알려준다.

In [9]:
KS, SEEDS = [1, 3, 5, 8, 16], [0, 1, 2]

def sample_kshot(pool, k, seed):
    order = pool.sample(frac=1, random_state=seed)
    has = {l: order['labels'].apply(lambda ls: l in ls) for l in TRAIN}
    chosen, cnt = [], Counter()
    for l in sorted(TRAIN, key=lambda l: has[l].sum()):
        for ix in order.index[has[l]]:
            if cnt[l] >= k: break
            if ix not in chosen:
                chosen.append(ix); cnt.update(order.at[ix, 'labels'])
    short = [l for l in TRAIN if cnt[l] < k]
    return pool.loc[chosen], cnt, short

def to_gliclass(d, lids):
    return [{'text': r.text, 'all_labels': [text_of[l] for l in lids],
             'true_labels': [text_of[l] for l in r.labels if l in lids]} for r in d.itertuples()]

for k in 'AB':
    pool = parts[k][parts[k].split == 'pool']
    out = SPLITS / k / 'kshot'
    shutil.rmtree(out, ignore_errors=True); out.mkdir(parents=True)
    print(f'===== {k}')
    for kk in KS:
        for sd in SEEDS:
            s, cnt, short = sample_kshot(pool, kk, sd)
            if short:
                print(f'  k={kk} seed={sd}: 만들지 않음 (부족: {[name[l] for l in short]})'); break
            json.dump(to_gliclass(s, TRAIN), open(out / f'train_k{kk}_s{sd}.json', 'w'), ensure_ascii=False)
            if sd == 0:
                print(f'  k={kk}: 윈도우 {len(s)}개, 라벨별 {dict(cnt)}')

===== A
  k=1: 윈도우 4개, 라벨별 {'TA0006': 1, 'TA0001+TA0005': 1, 'TA0003+TA0004': 1, 'TA0007': 1, 'BENIGN': 1}
  k=3: 윈도우 12개, 라벨별 {'TA0006': 3, 'TA0001+TA0005': 3, 'TA0003+TA0004': 3, 'TA0007': 3, 'BENIGN': 3}
  k=5: 윈도우 20개, 라벨별 {'TA0006': 5, 'TA0001+TA0005': 5, 'TA0003+TA0004': 5, 'TA0007': 5, 'BENIGN': 5}
  k=8: 윈도우 32개, 라벨별 {'TA0006': 8, 'TA0001+TA0005': 8, 'TA0003+TA0004': 8, 'TA0007': 8, 'BENIGN': 8}


  k=16: 윈도우 64개, 라벨별 {'TA0006': 16, 'TA0001+TA0005': 16, 'TA0003+TA0004': 16, 'TA0007': 16, 'BENIGN': 16}
===== B
  k=1: 윈도우 4개, 라벨별 {'TA0006': 1, 'TA0001+TA0005': 1, 'TA0003+TA0004': 1, 'TA0007': 1, 'BENIGN': 1}
  k=3: 윈도우 12개, 라벨별 {'TA0006': 3, 'TA0001+TA0005': 3, 'TA0003+TA0004': 3, 'TA0007': 3, 'BENIGN': 3}
  k=5: 윈도우 20개, 라벨별 {'TA0006': 5, 'TA0001+TA0005': 5, 'TA0003+TA0004': 5, 'TA0007': 5, 'BENIGN': 5}
  k=8 seed=0: 만들지 않음 (부족: ['Credential Access'])
  k=16 seed=0: 만들지 않음 (부족: ['Credential Access'])


## 9. 나눈 결과 + 평가 파일 저장 (04의 7번과 같음)

In [10]:
cols = ['text', 'labels', 'session', 'window', 'source', 'split']
for k in 'AB':
    out = SPLITS / k
    for sp in ['pool', 'val', 'test']:
        parts[k][parts[k].split == sp][cols].to_json(out / f'{sp}.jsonl', orient='records', lines=True, force_ascii=False)
    json.dump(to_gliclass(parts[k][parts[k].split == 'val'], TRAIN), open(out / 'val_gliclass.json', 'w'), ensure_ascii=False)
    json.dump(to_gliclass(parts[k][parts[k].split == 'test'], TRAIN + HELD), open(out / 'test_gliclass.json', 'w'), ensure_ascii=False)
    print(k, sorted(os.listdir(out)), '| kshot 파일', len(os.listdir(out / 'kshot')), '개')

ex = json.load(open(SPLITS / 'B' / 'kshot' / 'train_k1_s0.json'))[0]
print('\n학습 파일 한 줄 예시:'); print(' text:', ex['text'][:150]); print(' true_labels:', [t[:40] for t in ex['true_labels']]); print(' all_labels 개수:', len(ex['all_labels']))
print('\n05에서 쓰려면: notebooks/finetuning.ipynb의 VARIANT를 A 또는 B로 두고 실행 (data/splits를 자동으로 찾음)')

A ['kshot', 'pool.jsonl', 'test.jsonl', 'test_gliclass.json', 'val.jsonl', 'val_gliclass.json'] | kshot 파일 15 개
B ['kshot', 'pool.jsonl', 'test.jsonl', 'test_gliclass.json', 'val.jsonl', 'val_gliclass.json'] | kshot 파일 9 개

학습 파일 한 줄 예시:
 text: CloudTrail window: 2 call groups by 2 actors (AssumedRole), 2 failed, region unchanged. Sequence: ec2 GetPasswordData (read, x30, failed: Client.Unaut
 true_labels: ['Credential Access: the actor tries to ob']
 all_labels 개수: 5

05에서 쓰려면: notebooks/finetuning.ipynb의 VARIANT를 A 또는 B로 두고 실행 (data/splits를 자동으로 찾음)
